# Urban Growth Prediction

In [ ]:
# import libraries
import pandas as pd
import ee
import geemap

In [ ]:
# Authenticate GEE
ee.Authenticate()

In [ ]:
# Initialize GEE
ee.Initialize()

In [ ]:
# Center coordinate to show map
fct_center = (9.056266, 7.498522)

# Boundary visualization params
vis_params_fao_1 = {
    "fillcolor":"b5ffb4", "color": "00909f",
    "width":1.0,
}
vis_params_aoi = {"fillcolor":"","color":"red"}
# Sentinel-2 visualization parameters
vis_params_s2_rgb = {"min":0, "max":0.3,"bands":["B4", "B3", "B2"]}
vis_params_s2_fcc = {"min":0, "max":0.3,"bands":["B8", "B4", "B3"]}

# Spectral indices visualization parameters
# NDVI
vis_params_ndvi = {
    "min":-0.2,
    "max" : 0.8,
    "palette":[
        "#a50026",
        "#d73027",
        "#f46d43",
        "#fdae61",
        "#fee08b",
        "#d9ef8b",
        "#86d96a",
        "#66bd63",
        "#1a9850",
        "#006837",
    ],
}




# NDWI
vis_params_ndwi = {
    "min":-0.5,
        "max": 0.5,
        "palette": [
            "#543005",
            "#8c510a",
            "#d8b365",
            "#f6e863",
            "#c7eae5",
            "#5ab4ac",
            "#01665e",


        ]

}
# NDBI

vis_params_ndbi = {
    "min": -0.5,
    "max": 0.5,
    "palette": [
        "#ffffff",  
        "#ffffff",  
        "#f0f4f8",  
        "#d9e6f2",  
        "#99badd",  
        "#4a7cb5",  
        "#0f4c81"   
    ]
}

In [ ]:

vis_params_Fao_1 = {
  "fillColor": 'b5ffb4',
  "color": '00909F',
  "width": 1.0,
}
vis_params_aoi =  {"fillColor":"", "color": "red"}
# Boundary visualization params 
vis_params_fao_1 = {
  "fillColor": 'b5ffb4',
  "color": '00909F',
  "width": 1.0,
}

vis_params_aoi = {"fillcolor": "", "color": "red"}

# Sentinel-2 Visualization parameters 
vis_params_s2_rgb = {"min" : 0, "max" :0.3, "bands": ["B4", "B3", "B2"]}
vis_params_s2_fcc = {"min" : 0, "max" :0.3, "bands": ["B8", "B4", "B3"]}



# Visualisation parameters for road layers
vis_params_roads_vector = {
                        "color": "red",
                        "width": 1.5,
                    }

vis_params_roads_raster = {
                        "min": 0,
                        "max": 1,
                        "palette": ["black", "white"],
                    }


vis_params_dist_road = {
    "min": 0,
    "max": 21730, #meters
    "palette": [
        "#d73027",  
        "#f46d43",
        "#fdae61",
        "#fee08b",
        "#d9ef8b",
        "#a6d96a",
        "#1a9850",  
    ],
}

# Visualisation parameters for water layers
vis_params_water = {
    "min": 0,
    "max": 1,
    "palette": ["white", "blue"], 
}

# Distance to water
vis_params_dist_water = {
    "min": 0,
    "max": 38497,  
    "palette": [
        "#f7fbff",  
        "#deebf7",
        "#9ecae1",
        "#4292c6",
        "#2171b5",
        "#08306b",  
    ],
}


# Nighttime lights visualisation parameters
vis_params_ntl = {
    "min": 0,
    "max": 20,  
    "palette": [
        "#000000",  
        "#2c0b00",
        "#6e1c00",
        "#a83800",
        "#d9720a",
        "#f7b733",
        "#ffe98a",  
    ],
}


# Visualization parameters for GPWv411 population DENSITY layer
vis_params_gpw = {
  "min": 0.0,
  "max": 10000.0,
  "palette": ["ffffe7", "FFc869", "ffac1d", "e17735", "f2552c", "9f0c21"]
  }

In [ ]:
# Boundary Data From FAO GAUL
# Data source: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_SIMPLIFIED_500m_2015_level0
fao_gaul_l0 = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level0') # Countries boundaries
fao_gaul_l1 = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level1') # States boundaries
fao_gaul_l2 = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level1') # LGAs boundaries





boundary_Map = geemap.Map(center=fct_center, zoom=10)
boundary_Map.addLayer(fao_gaul_l0, {}, 'Country Boundaries')
boundary_Map.addLayer(fao_gaul_l1, {}, 'State Boundaries')
boundary_Map.addLayer(fao_gaul_l2, {}, 'LGA Boundaries')
boundary_Map

In [ ]:
# Select just a single feature
print(fao_gaul_l0.limit(1).getInfo()["columns"])
nga_l0 = fao_gaul_l0.filter(ee.Filter.eq("ADM0_NAME", "Nigeria"))
nga_l1 = fao_gaul_l1.filter(ee.Filter.eq("ADM0_NAME", "Nigeria"))
fct_l0 = nga_l1.filter(ee.Filter.eq("ADM1_NAME", "Abuja"))
print(nga_l0.getInfo())
#get geometry of Abuja Boundary Feature Collection
aoi = fct_l0.geometry()
aoi_bbox = aoi.bounds()

#Create a map to visualize Abuja Boundary
aoi_map = geemap.Map(center=fct_center, zoom=10)

# 7. Add layer (Fixed capital 'L' in addLayer)
aoi_map.addLayer(fct_l0, vis_params_aoi, 'Abuja Boundary')
aoi_map


In [ ]:
# Permanent water (JRC Global Surface Water)
# Data source: https://developers.google.com/earth-engine/datasets/catalog/JRC_GSW1_4_GlobalSurfaceWater
gsw = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').clip(fct_l0.geometry())
permanent_water = gsw.select('occurrence').gte(50) # >= 50% the time the water present

# Compute Euclidean distance to permanent water
distance_to_water = (
    permanent_water.Not()
    .fastDistanceTransform(256)
    .sqrt()
    .multiply(ee.Image.pixelArea().sqrt()) #convert to meters
    .rename('dist_to_water')
    .clip(fct_l0.geometry())
)

# Maximum distance 'distance_to_water' layer
print(distance_to_water.reduceRegion(ee.Reducer.max(), fct_l0.geometry(), 1000, maxPixels=1e9).getInfo())

thematic_map_2 = geemap.Map(center=fct_center, zoom=8)
thematic_map_2.add_basemap('SATELLITE')
thematic_map_2.addLayer(distance_to_water, vis_params_water, 'Permanent Water')
thematic_map_2.addLayer(distance_to_water.select('dist_to_water'), vis_params_dist_water, 'Distance to Water')

thematic_map_2

In [ ]:
# population density (CIESIN GPWv4.11)
# Data source: https://developers.google.com/earth-engine/datasets/catalog/CIESIN_GPWv411_population_density
# Closest available year to analysis baseline (2015/2020/2025)
gpw = ee.ImageCollection('CIESIN/GPWv411/GPW_Population_Density')

def get_population_density(year, extent):
    ''' Return the GPWv411 population density image closest to the given year.'''
    available_years = [2000, 2005, 2010, 2015, 2020]
    closest_year = min(available_years, key=lambda y: abs(y - year))
    image = ( 
        gpw.filter(ee.Filter.calendarRange(closest_year, closest_year, "year"))
        .first()
        .select('population_density')
        .rename("pop_density")
        .clip(extent)
        
    )
    return image
# Apply the function
pop_density_2015 = get_population_density(2015, fct_l0.geometry())
pop_density_2020 = get_population_density(2020, fct_l0.geometry())
pop_density_2022 = get_population_density(2025, fct_l0.geometry())

thematic_map_4 = geemap.Map(center=fct_center, zoom=8)
thematic_map_4.add_basemap('SATELLITE')
thematic_map_4.addLayer(pop_density_2015, vis_params_gpw, 'population density - 2015')
thematic_map_4.addLayer(pop_density_2020, vis_params_gpw, 'population density - 2020')
thematic_map_4.addLayer(pop_density_2022, vis_params_gpw, 'population density - 2022 (uses 2022 data)')
thematic_map_4
  